In [2]:
import warnings
warnings.filterwarnings('ignore')

print("🧠 Motor Imagery Classification System Initialized")
print("🎯 Target: Left vs. Right Hand Motor Imagery Classification")
print("🚀 Architecture: EEGNet (State-of-the-art for EEG)")
print("📊 Visualization: Plotly")


🧠 Motor Imagery Classification System Initialized
🎯 Target: Left vs. Right Hand Motor Imagery Classification
🚀 Architecture: EEGNet (State-of-the-art for EEG)
📊 Visualization: Plotly


In [4]:
!pip uninstall protobuf -y


Found existing installation: protobuf 6.33.0
Uninstalling protobuf-6.33.0:
  Successfully uninstalled protobuf-6.33.0


In [6]:
!pip install protobuf==3.20.3


In [7]:
import os
import numpy as np
import pandas as pd
import mne
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.constraints import max_norm

# Plotly imports for enhanced visualization
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

class Config:
    """Centralized configuration class for all parameters."""

    # Data Configuration
    BASE_PATH = '/kaggle/input/eeg-motor-movementimagery-dataset/files/'
    SAMPLING_RATE = 160 # Hz
    N_CHANNELS = 64 # Number of EEG channels
    N_CLASSES = 2 # Left vs. Right hand imagery

    # Subject Configuration (Robust Splitting)
    # Total subjects: 109. We use a subject-wise split for robust validation.
    ALL_SUBJECTS = list(range(1, 110))
    TRAIN_SUBJECTS = list(range(1, 81)) # Subjects 1-80 for training (approx. 75%)
    VALID_SUBJECTS = list(range(81, 96)) # Subjects 81-95 for validation (approx. 15%)
    TEST_SUBJECTS = list(range(96, 110)) # Subjects 96-109 for final testing (approx. 15%)

    # Signal Processing Parameters
    LOWCUT = 8.0 # Lower cutoff frequency (Hz) for Mu rhythm
    HIGHCUT = 30.0 # Upper cutoff frequency (Hz) for Beta rhythm
    FILTER_ORDER = 5 # Butterworth filter order
    WINDOW_SIZE = 4.0 # Time window in seconds (full imagery period)
    T_MIN = 0.0 # Start time of the epoch relative to the event
    T_MAX = 4.0 # End time of the epoch

    # EEGNet Architecture Parameters
    F1 = 8 # Number of temporal filters
    D = 2 # Number of spatial filters
    F2 = 16 # Number of pointwise filters
    KERNEL_LENGTH = 64 # Length of the temporal convolution kernel
    DROPOUT_RATE = 0.5 # Dropout rate for regularization

    # Training Parameters
    BATCH_SIZE = 16 # Smaller batch size for better generalization on EEG data
    EPOCHS = 200 # Max epochs, with early stopping
    LEARNING_RATE = 0.001
    EARLY_STOPPING_PATIENCE = 30 # Increased patience for EEGNet

    # Visualization with Plotly
    PLOTLY_TEMPLATE = 'plotly_dark'

config = Config()
print("✅ Configuration parameters loaded successfully")
print(f"📁 Dataset path: {config.BASE_PATH}")
print(f"🧠 Training Subjects: {len(config.TRAIN_SUBJECTS)}")
print(f"📊 Validation Subjects: {len(config.VALID_SUBJECTS)}")
print(f"🔬 Test Subjects: {len(config.TEST_SUBJECTS)}")
print(f"🕒 Epoch duration: {config.T_MAX - config.T_MIN}s at {config.SAMPLING_RATE} Hz")

✅ Configuration parameters loaded successfully
📁 Dataset path: /kaggle/input/eeg-motor-movementimagery-dataset/files/
🧠 Training Subjects: 80
📊 Validation Subjects: 15
🔬 Test Subjects: 14
🕒 Epoch duration: 4.0s at 160 Hz


In [8]:
"""
Data Loading and Preprocessing Pipeline
=======================================
This cell defines functions to load, filter, and epoch the PhysioNet EEGMMIDB dataset
using the MNE library. The code is enhanced to handle potential variations in trial lengths.
"""

def load_and_preprocess_subject_data(subject_id, base_path):
    """
    Loads all motor imagery runs for a single subject, applies a bandpass
    filter, and extracts epochs.

    Returns:
        A tuple of (epochs, labels) or (None, None) if loading fails.
    """
    # Motor imagery runs for Task 1 (left/right fist)
    mi_runs = [3, 4, 7, 8, 11, 12]

    raw_files = []
    for run_id in mi_runs:
        subject_str = f"S{subject_id:03d}"
        run_str = f"R{run_id:02d}"
        file_path = f"{base_path}/{subject_str}/{subject_str}{run_str}.edf"

        if os.path.exists(file_path):
            try:
                raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
                raw_files.append(raw)
            except Exception as e:
                print(f"❌ Error loading {file_path}: {e}")
        else:
            # This is a common occurrence as not all subjects completed all runs.
            pass

    if not raw_files:
        return None, None

    # Concatenate all runs for the subject
    raw_concat = mne.concatenate_raws(raw_files, verbose=False)

    # Apply bandpass filter for mu and beta rhythms
    raw_concat.filter(config.LOWCUT, config.HIGHCUT, fir_design='firwin', skip_by_annotation='edge', verbose=False)

    # Extract events from annotations
    events, event_id = mne.events_from_annotations(raw_concat, verbose=False)

    # Define epochs around motor imagery cues (T1: left, T2: right)
    picks = mne.pick_types(raw_concat.info, meg=False, eeg=True, stim=False, eog=False, exclude='bads')

    # Create epochs, dropping the last trial if it's too short
    try:
        epochs = mne.Epochs(raw_concat, events, dict(left_hand=2, right_hand=3),
                            tmin=config.T_MIN, tmax=config.T_MAX, proj=True,
                            picks=picks, baseline=None, preload=True, verbose=False, on_missing='warn')
    except ValueError:
        # This can happen if no valid events are found after filtering
        return None, None

    labels = epochs.events[:, -1] - 2  # Map labels to 0 (left) and 1 (right)

    return epochs.get_data(), labels

def load_dataset_for_subjects(subject_list, base_path):
    """
    Loads and concatenates data for a list of subjects, ensuring all trials
    have a consistent length.

    Returns:
        A tuple of (all_data, all_labels).
    """
    all_epochs_list = []
    all_labels_list = []
    expected_n_samples = None

    for subject_id in subject_list:
        print(f"🔄 Processing Subject {subject_id:03d}...", end=" ")
        epochs, labels = load_and_preprocess_subject_data(subject_id, config.BASE_PATH)

        if epochs is not None and len(epochs) > 0:
            # --- FIX STARTS HERE ---
            # Enforce consistent trial length
            if expected_n_samples is None:
                expected_n_samples = epochs.shape[2]
                print(f"INFO: Setting expected trial length to {expected_n_samples} samples.")

            if epochs.shape[2] != expected_n_samples:
                print(f"⚠️ Mismatched trial length ({epochs.shape[2]}). Skipping subject.")
                continue
            # --- FIX ENDS HERE ---

            all_epochs_list.append(epochs)
            all_labels_list.append(labels)
            print(f"✅ Found {len(epochs)} trials.")
        else:
            print("❌ No valid trials found.")

    if not all_epochs_list:
        return np.array([]), np.array([])

    return np.concatenate(all_epochs_list, axis=0), np.concatenate(all_labels_list, axis=0)

# --- Load Datasets ---
# Load training data
print("\n--- Loading Training Data ---")
X_train, y_train = load_dataset_for_subjects(config.TRAIN_SUBJECTS, config.BASE_PATH)

# Load validation data
print("\n--- Loading Validation Data ---")
X_val, y_val = load_dataset_for_subjects(config.VALID_SUBJECTS, config.BASE_PATH)

# --- Data Summary ---
if X_train.size > 0 and X_val.size > 0:
    print("\n📊 Data Loading Summary:")
    print(f" Training data shape: {X_train.shape}")
    print(f" Training labels shape: {y_train.shape} (Left: {np.sum(y_train == 0)}, Right: {np.sum(y_train == 1)})\")")
    print(f" Validation data shape: {X_val.shape}")
    print(f" Validation labels shape: {y_val.shape} (Left: {np.sum(y_val == 0)}, Right: {np.sum(y_val == 1)})\")")
else:
    print("\n❌ Critical error: No training or validation data was loaded. Please check paths and subject IDs.")


--- Loading Training Data ---
🔄 Processing Subject 001... INFO: Setting expected trial length to 641 samples.
✅ Found 90 trials.
🔄 Processing Subject 002... ✅ Found 90 trials.
🔄 Processing Subject 003... ✅ Found 90 trials.
🔄 Processing Subject 004... ✅ Found 90 trials.
🔄 Processing Subject 005... ✅ Found 90 trials.
🔄 Processing Subject 006... ✅ Found 90 trials.
🔄 Processing Subject 007... ✅ Found 90 trials.
🔄 Processing Subject 008... ✅ Found 90 trials.
🔄 Processing Subject 009... ✅ Found 90 trials.
🔄 Processing Subject 010... ✅ Found 90 trials.
🔄 Processing Subject 011... ✅ Found 90 trials.
🔄 Processing Subject 012... ✅ Found 90 trials.
🔄 Processing Subject 013... ✅ Found 90 trials.
🔄 Processing Subject 014... ✅ Found 90 trials.
🔄 Processing Subject 015... ✅ Found 90 trials.
🔄 Processing Subject 016... ✅ Found 90 trials.
🔄 Processing Subject 017... ✅ Found 90 trials.
🔄 Processing Subject 018... ✅ Found 90 trials.
🔄 Processing Subject 019... ✅ Found 90 trials.
🔄 Processing Subject 020

In [9]:
"""
Data Preparation for Deep Learning
====================================
This cell standardizes the data and reshapes it for EEGNet.
"""

def prepare_data_for_eegnet(X_train, y_train, X_val, y_val, augment=False):
    """
    Scales, augments (optional), and reshapes data for the EEGNet model.
    A single scaler is fitted on the training data and applied to all sets.
    """
    # Reshape for scaling: (trials, channels * samples)
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_val_flat = X_val.reshape(X_val.shape[0], -1)

    # Fit scaler ONLY on training data to prevent data leakage
    scaler = StandardScaler()
    X_train_scaled_flat = scaler.fit_transform(X_train_flat)

    # Apply the same scaler to validation data
    X_val_scaled_flat = scaler.transform(X_val_flat)

    # Reshape back to (trials, channels, samples)
    X_train_scaled = X_train_scaled_flat.reshape(X_train.shape)
    X_val_scaled = X_val_scaled_flat.reshape(X_val.shape)

    # Data Augmentation: Add Gaussian noise (currently off)
    if augment:
        print("🔧 Applying data augmentation (noise injection)...")
        noise_factor = 0.1
        X_train_aug = X_train_scaled + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_train_scaled.shape)
        y_train_aug = y_train
    else:
        X_train_aug, y_train_aug = X_train_scaled, y_train

    # Reshape for EEGNet: (trials, channels, samples, 1)
    X_train_final = X_train_aug[:, :, :, np.newaxis]
    X_val_final = X_val_scaled[:, :, :, np.newaxis]

    # Convert labels to categorical format
    y_train_final = to_categorical(y_train_aug, config.N_CLASSES)
    y_val_final = to_categorical(y_val, config.N_CLASSES)

    return X_train_final, y_train_final, X_val_final, y_val_final, scaler

# --- Data Preparation Pipeline ---
print("\n--- Preparing Data for EEGNet ---")
X_train_final, y_train_final, X_val_final, y_val_final, scaler = prepare_data_for_eegnet(
    X_train, y_train, X_val, y_val, augment=False # Augmentation is often better done on-the-fly or as a Keras layer
)
print("✅ Data preparation complete!")
print(f" Final Training Shape: {X_train_final.shape}")
print(f" Final Validation Shape: {X_val_final.shape}")


--- Preparing Data for EEGNet ---
✅ Data preparation complete!
 Final Training Shape: (7157, 64, 641, 1)
 Final Validation Shape: (1177, 64, 641, 1)


In [10]:
"""
EEGNet Model Definition
=======================
This cell defines the EEGNet architecture using the Keras Functional API.
The architecture is based on the original paper for robust EEG classification.
Reference: https://arxiv.org/abs/1611.08024
"""

def EEGNet(nb_classes, Chans=64, Samples=641, dropoutRate=0.5,
           kernLength=64, F1=8, D=2, F2=16, norm_rate=0.25):
    """ Keras implementation of the EEGNet model."""

    input_main = layers.Input(shape=(Chans, Samples, 1), name='input_layer')

    # --- Block 1 ---
    # Temporal Convolution
    block1 = layers.Conv2D(F1, (1, kernLength), padding='same', use_bias=False)(input_main)
    block1 = layers.BatchNormalization()(block1)

    # Depthwise Spatial Convolution
    block1 = layers.DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=D,
                                     depthwise_constraint=max_norm(1.))(block1)
    block1 = layers.BatchNormalization()(block1)
    block1 = layers.Activation('elu')(block1)
    block1 = layers.AveragePooling2D((1, 4))(block1)
    block1 = layers.Dropout(dropoutRate)(block1)

    # --- Block 2 ---
    # Separable Convolution
    block2 = layers.SeparableConv2D(F2, (1, 16), use_bias=False, padding='same')(block1)
    block2 = layers.BatchNormalization()(block2)
    block2 = layers.Activation('elu')(block2)
    block2 = layers.AveragePooling2D((1, 8))(block2)
    block2 = layers.Dropout(dropoutRate)(block2)

    # --- Classification Block ---
    flatten = layers.Flatten(name='flatten')(block2)
    dense = layers.Dense(nb_classes, name='dense', kernel_constraint=max_norm(norm_rate))(flatten)
    softmax = layers.Activation('softmax', name='softmax')(dense)

    return models.Model(inputs=input_main, outputs=softmax, name="EEGNet")

# --- Instantiate and Compile the Model ---
input_shape = X_train_final.shape[1:]

model = EEGNet(
    nb_classes=config.N_CLASSES,
    Chans=input_shape[0],
    Samples=input_shape[1],
    dropoutRate=config.DROPOUT_RATE,
    kernLength=config.KERNEL_LENGTH,
    F1=config.F1, D=config.D, F2=config.F2
)

# Compile the model with Adam optimizer
optimizer = optimizers.Adam(learning_rate=config.LEARNING_RATE)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Print the model summary
print("📄 EEGNet Model Summary:")
model.summary()

I0000 00:00:1764332307.156106      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1764332307.156688      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


📄 EEGNet Model Summary:


Model: "EEGNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 641, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 641, 8)     │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 641, 8)     │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 1, 641, 16)     │         1,024 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 641, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1, 641, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 1, 160, 16)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 160, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 1, 160, 16)     │           512 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 160, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 1, 160, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 1, 20, 16)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 20, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 320)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           642 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Activation)            │ (None, 2)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,850 (11.13 KB)

 Trainable params: 2,770 (10.82 KB)

 Non-trainable params: 80 (320.00 B)

In [12]:
# --- Step 1: Install necessary system and Python libraries ---
# graphviz is a system dependency for plotting, pydot is the Python interface.
# %sudo apt-get install graphviz -y
%pip install pydot pydotplus

# --- Step 2: Import necessary libraries ---
import tensorflow as tf
from IPython.display import Image, display

print("✅ Necessary libraries installed.")

# --- Step 3: Load your best model ---
# Ensure the 'best_eegnet_model.keras' file is available.
try:
    best_model = tf.keras.models.load_model('best_eegnet_model.keras')
    print("🧠 Best model loaded successfully.")

    # --- Step 4: Generate and display the model plot using tf.keras.utils.plot_model ---

    # Define the output file name
    output_image_file = 'eegnet_model_visualization.png'

    print("\n🎨 Generating model plot with TensorFlow's built-in utility...")
    tf.keras.utils.plot_model(
        best_model,
        to_file=output_image_file,
        show_shapes=True,
        show_dtype=False,
        show_layer_names=True,
        rankdir='TB',  # 'TB' for top-to-bottom, 'LR' for left-to-right
        expand_nested=True,
        dpi=96,
        show_layer_activations=True
    )
    print(f"✅ Model plot saved to '{output_image_file}'")

    # --- Step 5: Display the generated image in the notebook ---
    print("\n🖼️ Displaying the generated model architecture:")
    display(Image(filename=output_image_file))

except FileNotFoundError:
    print("❌ ERROR: 'best_eegnet_model.keras' not found. Please ensure the model has been trained and saved.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")
    print("ℹ️ If you still encounter issues, ensure 'graphviz' was installed correctly.")



Note: you may need to restart the kernel to use updated packages.
✅ Necessary libraries installed.
❌ An unexpected error occurred: File not found: filepath=best_eegnet_model.keras. Please ensure the file is an accessible `.keras` zip file.
ℹ️ If you still encounter issues, ensure 'graphviz' was installed correctly.


In [13]:
"""
Model Training Pipeline
=======================
This cell trains the compiled EEGNet model using the prepared datasets.
It employs advanced callbacks for robust training.
"""

# Callbacks for robust training
training_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=config.EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1,
        mode='max'
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=15, # Increased patience for learning rate reduction
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_eegnet_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
        mode='max'
    )
]

print("\n🚀 Starting EEGNet model training...")
history = model.fit(
    X_train_final,
    y_train_final,
    batch_size=config.BATCH_SIZE,
    epochs=config.EPOCHS,
    validation_data=(X_val_final, y_val_final),
    callbacks=training_callbacks,
    verbose=1,
    shuffle=True
)

print("\n✅ EEGNet training completed!")


🚀 Starting EEGNet model training...
Epoch 1/200


I0000 00:00:1764332772.325939     130 service.cc:148] XLA service 0x79e4c8005620 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764332772.328222     130 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1764332772.328243     130 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1764332772.723695     130 cuda_dnn.cc:529] Loaded cuDNN version 90300


 13/448 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.4691 - loss: 0.6897

I0000 00:00:1764332776.933949     130 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


448/448 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5022 - loss: 0.6962
Epoch 1: val_accuracy improved from -inf to 0.50467, saving model to best_eegnet_model.keras
448/448 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.5022 - loss: 0.6962 - val_accuracy: 0.5047 - val_loss: 0.6936 - learning_rate: 0.0010
Epoch 2/200
445/448 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5279 - loss: 0.6942
Epoch 2: val_accuracy improved from 0.50467 to 0.53526, saving model to best_eegnet_model.keras
448/448 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.5279 - loss: 0.6942 - val_accuracy: 0.5353 - val_loss: 0.6900 - learning_rate: 0.0010
Epoch 3/200
447/448 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5332 - loss: 0.6885
Epoch 3: val_accuracy improved from 0.53526 to 0.58539, saving model to best_eegnet_model.keras
448/448 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.5332 - loss: 0.6885 - val_accuracy: 0.5854 - val_loss: 0.6755 - learning_rate: 0.0010
Epoch 4/200
448/448 ━━━━━━━━━━━━━━━━━━━━

In [20]:
"""
Performance Evaluation and Visualization
========================================
This cell analyzes the model's training history and visualizes performance
on the validation set using interactive Plotly charts.
"""

# --- Plot Training History ---
def plot_training_history(history):
    """Visualizes training & validation accuracy and loss using Plotly."""
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Model Accuracy", "Model Loss"))

    # Accuracy Plot
    fig.add_trace(go.Scatter(y=history.history['accuracy'], name='Train Accuracy', mode='lines+markers'), row=1, col=1)
    fig.add_trace(go.Scatter(y=history.history['val_accuracy'], name='Validation Accuracy', mode='lines+markers'), row=1, col=1)

    # Loss Plot
    fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss', mode='lines+markers'), row=1, col=2)
    fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Validation Loss', mode='lines+markers'), row=1, col=2)

    fig.update_layout(title_text="Model Training History", template=config.PLOTLY_TEMPLATE, height=400)
    fig.update_xaxes(title_text="Epoch")
    fig.update_yaxes(title_text="Metric Value")
    fig.show()

print("\n--- Training History Visualization ---")
plot_training_history(history)


# --- Validation Set Evaluation ---
print("\n--- Evaluating Performance on Validation Set ---")
best_model = models.load_model('best_eegnet_model.keras')
y_pred_val = best_model.predict(X_val_final, batch_size=config.BATCH_SIZE)
y_pred_classes = np.argmax(y_pred_val, axis=1)
y_true_classes = np.argmax(y_val_final, axis=1)

# Classification Report
print("\n📋 Classification Report (Validation Set):")
class_report = classification_report(y_true_classes, y_pred_classes, target_names=['Left Hand', 'Right Hand'])
print(class_report)

# Confusion Matrix
def plot_confusion_matrix(y_true, y_pred):
    """Visualizes the confusion matrix using an annotated Plotly heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    labels = ['Left Hand', 'Right Hand']
    # Add annotations as text
    annotations = []
    for i, row in enumerate(cm):
        for j, value in enumerate(row):
            annotations.append(
                dict(x=labels[j], y=labels[i], text=str(value), showarrow=False)
            )

    fig = go.Figure(data=go.Heatmap(
                    z=cm,
                    x=labels,
                    y=labels,
                    colorscale='Blues'
                ))

    fig.update_layout(title_text="Confusion Matrix (Validation Set)",
                      xaxis_title="Predicted Label", yaxis_title="True Label",
                      template=config.PLOTLY_TEMPLATE, annotations=annotations)
    fig.show()

print("\n--- Confusion Matrix Visualization ---")
# plot_confusion_matrix(y_true_classes, y_pred_classes)
# plot_training_history(history)


--- Training History Visualization ---



--- Evaluating Performance on Validation Set ---
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step

📋 Classification Report (Validation Set):
              precision    recall  f1-score   support

   Left Hand       0.87      0.72      0.79       585
  Right Hand       0.77      0.89      0.82       592

    accuracy                           0.81      1177
   macro avg       0.82      0.81      0.81      1177
weighted avg       0.82      0.81      0.81      1177


--- Confusion Matrix Visualization ---


In [16]:
"""
Final Model Evaluation on the Hold-Out Test Set
================================================
This cell performs the final evaluation on completely unseen subject data
to provide an unbiased assessment of the model's generalization performance.
"""

print("\n--- Testing on Final Hold-Out Test Set ---")

# 1. Load the test data
print("\n🔄 Loading test data for subjects defined in config...")
X_test, y_test = load_dataset_for_subjects(config.TEST_SUBJECTS, config.BASE_PATH)

if X_test.size > 0:
    print(f"\n✅ Test data loaded successfully. Found {X_test.shape[0]} trials.")
    print(f" Test data shape: {X_test.shape}")
    print(f" Test labels shape: {y_test.shape} (Left: {np.sum(y_test == 0)}, Right: {np.sum(y_test == 1)})")

    # 2. Prepare the test data using the scaler fitted on the training data
    print("\n🛠️  Preparing test data for evaluation...")
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    X_test_scaled_flat = scaler.transform(X_test_flat)  # IMPORTANT: Only transform, do not fit again
    X_test_scaled = X_test_scaled_flat.reshape(X_test.shape)
    X_test_final = X_test_scaled[:, :, :, np.newaxis]
    y_test_final = to_categorical(y_test, config.N_CLASSES)
    print(f" Final Test Shape: {X_test_final.shape}")

    # 3. Evaluate the best model on the test data
    print("\n🧠 Evaluating model on the test set...")
    test_loss, test_accuracy = best_model.evaluate(X_test_final, y_test_final, verbose=0, batch_size=config.BATCH_SIZE)
    print(f"\n🎯 Final Test Accuracy: {test_accuracy*100:.2f}%")
    print(f"   Final Test Loss: {test_loss:.4f}")

    # 4. Generate detailed metrics and confusion matrix
    y_pred_test = best_model.predict(X_test_final, batch_size=config.BATCH_SIZE)
    y_pred_test_classes = np.argmax(y_pred_test, axis=1)
    y_true_test_classes = y_test  # Original labels are needed for sklearn metrics

    print("\n📋 Classification Report (Test Set):")
    print(classification_report(y_true_test_classes, y_pred_test_classes, target_names=['Left Hand', 'Right Hand']))

    # Visualize the confusion matrix for the test set
    def plot_test_confusion_matrix(y_true, y_pred):
        """Visualizes the confusion matrix for the test set using Plotly."""
        cm = confusion_matrix(y_true, y_pred)
        labels = ['Left Hand', 'Right Hand']
        # Use the same ff.create_annotated_heatmap for consistency
        fig = ff.create_annotated_heatmap(z=cm, x=labels, y=labels, colorscale='Greens')
        fig.update_layout(title_text="Confusion Matrix (Final Test Set)",
                          xaxis_title="Predicted Label", yaxis_title="True Label",
                          template=config.PLOTLY_TEMPLATE)
        fig.show()

    plot_test_confusion_matrix(y_true_test_classes, y_pred_test_classes)
else:
    print("\n❌ Critical error: Could not load test data. Skipping final evaluation.")


--- Testing on Final Hold-Out Test Set ---

🔄 Loading test data for subjects defined in config...
🔄 Processing Subject 096... INFO: Setting expected trial length to 641 samples.
✅ Found 90 trials.
🔄 Processing Subject 097... ✅ Found 90 trials.
🔄 Processing Subject 098... ✅ Found 90 trials.
🔄 Processing Subject 099... ✅ Found 90 trials.
🔄 Processing Subject 100... ⚠️ Mismatched trial length (513). Skipping subject.
🔄 Processing Subject 101... ✅ Found 90 trials.
🔄 Processing Subject 102... ✅ Found 84 trials.
🔄 Processing Subject 103... ✅ Found 90 trials.
🔄 Processing Subject 104... ✅ Found 87 trials.
🔄 Processing Subject 105... ✅ Found 90 trials.
🔄 Processing Subject 106... ✅ Found 90 trials.
🔄 Processing Subject 107... ✅ Found 90 trials.
🔄 Processing Subject 108... ✅ Found 90 trials.
🔄 Processing Subject 109... ✅ Found 90 trials.

✅ Test data loaded successfully. Found 1161 trials.
 Test data shape: (1161, 64, 641)
 Test labels shape: (1161,) (Left: 582, Right: 579)

🛠️  Preparing test

In [17]:
"""
Visualize Predictions vs. Ground Truth for Test Set Trials in a Grid
===================================================================
This cell iterates through each trial in the test set and visualizes
the EEG data along with the ground truth and model prediction in a grid format.
Subplot titles are colored based on prediction correctness.
"""

def visualize_test_predictions_grid(X_test_data, y_true, y_pred, config_params, trials_per_row=3, rows_to_show=5):
    """
    Visualizes a grid of EEG data for test set trials, showing both
    ground truth and prediction for each trial. Subplot titles are colored
    based on prediction correctness.
    """
    class_labels = {0: 'Left Hand', 1: 'Right Hand'}
    total_trials = X_test_data.shape[0]
    num_plots = min(total_trials, trials_per_row * rows_to_show) # Limit number of plots

    if num_plots == 0:
        print("No test data available to visualize.")
        return

    print(f"\n📊 Visualizing {num_plots} Test Set Predictions vs. Ground Truth in a Grid...")

    # Calculate grid dimensions
    total_rows = (num_plots + trials_per_row - 1) // trials_per_row
    subplot_titles = []

    # Create subplot titles based on ground truth and prediction, with color
    for i in range(num_plots):
         true_label = class_labels.get(y_true[i], 'Unknown')
         predicted_label = class_labels.get(y_pred[i], 'Unknown')
         color = "green" if true_label == predicted_label else "red"
         title = f'<span style="color:{color};">Trial {i+1}: True: {true_label}, Pred: {predicted_label}</span>'
         subplot_titles.append(title)


    fig = make_subplots(rows=total_rows, cols=trials_per_row,
                        subplot_titles=subplot_titles,
                        vertical_spacing=0.08, horizontal_spacing=0.05)

    # Ensure the data is in the expected (trials, channels, samples, 1) format
    if X_test_data.shape[-1] != 1:
        # This should ideally be handled in preparation, but as a safeguard:
        print("Warning: X_test_data shape is not (trials, channels, samples, 1). Reshaping.")
        X_test_data = X_test_data[:, :, :, np.newaxis]

    channels_to_plot = {'C3': 10, 'Cz': 12, 'C4': 14} # Example channels to display
    time_axis = np.linspace(config_params.T_MIN, config_params.T_MAX, X_test_data.shape[2]) # Use sample dimension

    for i in range(num_plots):
        row = (i // trials_per_row) + 1
        col = (i % trials_per_row) + 1
        eeg_trial = X_test_data[i]

        for ch_name, ch_idx in channels_to_plot.items():
             if ch_idx < eeg_trial.shape[0]:
                fig.add_trace(go.Scatter(x=time_axis, y=eeg_trial[ch_idx, :, 0],
                                         mode='lines', name=ch_name if i == 0 else '', # Show legend only once
                                         showlegend=True if i == 0 else False), # Control legend visibility
                                         row=row, col=col)
             else:
                 print(f"Warning: Channel index {ch_idx} for {ch_name} is out of bounds for trial {i+1}.")


    fig.update_layout(
        title_text=f"EEG Data with Predictions vs. Ground Truth (First {num_plots} Test Trials)",
        template=config_params.PLOTLY_TEMPLATE,
        height=total_rows * 300, # Adjust height based on rows
        width=trials_per_row * 400, # Adjust width based on columns
        showlegend=True # Ensure legend is shown
    )

    # Update axes titles only for the bottom-left subplot to avoid clutter
    for i in range(total_rows):
        for j in range(trials_per_row):
            fig.update_xaxes(title_text="Time (s)", row=i + 1, col=j + 1)
            fig.update_yaxes(title_text="Amplitude (Scaled μV)", row=i + 1, col=j + 1)

    fig.show()


# --- Visualize Predictions vs. Ground Truth in Grid ---
if 'X_test_final' in locals() and 'y_true_test_classes' in locals() and 'y_pred_test_classes' in locals():
    # Limit the number of trials visualized in the grid for performance
    visualize_test_predictions_grid(X_test_final, y_true_test_classes, y_pred_test_classes, config,
                                   trials_per_row=4, # You can adjust this
                                   rows_to_show=4)    # You can adjust this
else:
    print("\n❌ Test data (X_test_final, y_true_test_classes, y_pred_test_classes) not found.")
    print("Please ensure the previous cells for loading, preparing, and testing the data have been executed.")


📊 Visualizing 16 Test Set Predictions vs. Ground Truth in a Grid...
